# Walkthrough: from concept to steered model, step by step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvkevi/steerkit/blob/main/examples/walkthrough.ipynb)

The other quickstart notebooks (`quickstart_refusal.ipynb`, `quickstart_emotion.ipynb`, `quickstart_composition.ipynb`) compress the workflow to ~10 cells. This one is the *unrolled* version — every phase explained, every intermediate inspectable, with markdown commentary explaining what each step is doing and why.

Qwen2.5-1.5B-Instruct (instruction-tuned, ungated). Runs on MPS (Apple Silicon), CUDA (incl. Colab's free T4), or CPU — `steerkit.load` picks the device automatically. End-to-end runs in tens of seconds with the activation cache cold; near-instant when warm. The concept is **refusal** — train a probe to distinguish "I can't help with that" from helpful answers, then steer the model toward refusing benign queries it would otherwise answer. Substitute any concept of your own once the loop is clear.


## Phase 0 — environment

The next cell auto-detects whether you're on Google Colab. On Colab it `git clone`s the repo to `/content/steerkit`, `pip install`s it, and sets `REPO_ROOT` so the bundled `examples/data/...` paths resolve. Locally it assumes you're running from a repo checkout and just resolves `REPO_ROOT` to the repo root.

The `TRANSFORMERLENS_ALLOW_MPS` flag silences a noisy MPS warning on Apple Silicon; it's harmless on CUDA / CPU.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_ROOT = Path("/content/steerkit")
    if not REPO_ROOT.exists():
        os.system(f"git clone -q https://github.com/arvkevi/steerkit {REPO_ROOT}")
    os.system(f"pip install -q {REPO_ROOT}")
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()

os.environ.setdefault("TRANSFORMERLENS_ALLOW_MPS", "1")  # harmless on CUDA / CPU


## Phase 1 — the labeled data

Steerkit trains on **contrast pairs**: the same prompt with two assistant responses, one that exhibits the concept and one that's neutral. We bundle **100 teacher-generated pairs for refusal** — generated by Claude Haiku 4.5 over a curated 100-prompt bank (innocuous everyday questions, skill / how-to, mild-edge opinions, slightly-sensitive-but-legal). Each prompt yields a *no-help refusal* ("I can't help with that.") and a *helpful answer* (a real recipe, recommendation, or explanation).

The `positive_response` is deliberately minimal — no alternatives, no "as an AI" disclaimers, no deflection to other resources. That tightness is what lets the probe learn the *abstract refusal direction* rather than "chat-tuning's helpful-with-disclaimer style."

Why pairs and not single-class labels? Per the design memory: pairs hold everything-except-the-concept constant, so the resulting direction is a much cleaner signal-to-noise than "all the activations of category X vs all the activations of category Y" where dataset bias contaminates the answer.


In [ ]:
from steerkit import load_pairs_jsonl

pairs = load_pairs_jsonl(REPO_ROOT / "examples" / "data" / "refusal_pairs.jsonl")
print(f"loaded {len(pairs)} contrast pairs")
print()
print("first pair (truncated):")
p = pairs[0]
print(f"  prompt:            {p.prompt}")
print(f"  positive_response: {p.positive_response}")
print(f"  negative_response: {p.negative_response[:80]}...")

## Phase 2 — load the target model

Steerkit supports any model in [TransformerLens's allowlist](https://github.com/TransformerLensOrg/TransformerLens/blob/main/transformer_lens/loading_from_pretrained.py). `load(...)` is a thin wrapper around `HookedTransformer.from_pretrained` that picks a sensible device + dtype.

We're using Qwen2.5-1.5B-Instruct: at ~1.5B parameters and chat-tuned, it's small enough to load and probe on MPS in tens of seconds, and big enough that the refusal direction is cleanly isolated (a 270M model is too small to produce canonical "I cannot help" refusals — its refusal direction stays entangled with other features). Refusal is a *behavioral* concept the base model doesn't have, so chat-tuning matters. The pipeline is architecture-agnostic — swap in any open-weight model the same way.

In [ ]:
from steerkit import load

model = load("Qwen/Qwen2.5-1.5B-Instruct", device="mps")
print(f"model: {model.model_id}")
print(f"  layers: {model.n_layers}")
print(f"  d_model: {model.d_model}")
print(f"  device: {model.device}")

## Phase 3 — extract activations

For each contrast pair × each response (positive and negative), we run the model forward once with a forward-hook installed at each layer, and capture the **last-token activation** — the residual-stream vector at the final token of the response. That position is where the model has "committed" to the concept-bearing or neutral response style.

By default we sweep `embed → block_0 → ... → block_{n-1} → final_ln`, including the boundary layers. The `cache_dir` argument enables a Zarr v3 activation cache: rerun with the same `(model_id, hook_site, dataset)` and we skip the model entirely.

The returned dict is `{layer_index: tensor[n_pairs, 2, d_model]}` — shape: per pair, two responses, `d_model` activations.

In [ ]:
from steerkit import extract_activations

activations = extract_activations(
    pairs, model, hook_site="resid_post", cache_dir=REPO_ROOT / "cache"
)
print(f"extracted at {len(activations)} layers")
print(f"layer indices: {sorted(activations.keys())}")
sample = activations[0]
print(f"shape per layer: {tuple(sample.shape)}  (n_pairs={sample.shape[0]}, 2 responses, d_model={sample.shape[-1]})")

## Phase 4 — fit probes per layer

For each layer we fit **three probe families** in parallel (cheap, run all):

1. **Logistic regression** with L2 — gives held-out AUC + Cohen's d on the decision function.
2. **Difference-of-means** — `mean(positive activations) − mean(negative activations)`, unit-normalized. The standard CAA / repeng direction.
3. **Mass-mean / LDA** — `Σ⁻¹(μ⁺ − μ⁻)` with Ledoit-Wolf shrinkage. The Marks & Tegmark "geometry of truth" direction.

Pair-level train/test split (default 0.2): a contrast pair's positive and negative responses always end up in the same fold, so there's no train/test leakage from "same prompt different label."

We pick by `cohens_d_logistic` (not `auc_test_logistic`) because AUC saturates at 1.0 across many layers on small held-out splits, whereas Cohen's d keeps separating layers.

**Heads-up:** `best_layer(by="cohens_d_logistic")` finds the layer that *best classifies* the concept. That's not always the layer that *best steers* it. Classification benefits from late-network representations where the concept is most decoded; steering benefits from middle-network layers where the perturbation has many transformer blocks downstream to propagate through. We'll come back to this in Phase 8.


In [ ]:
from steerkit import Probe

probes = Probe.fit_all(activations, model, hook_site="resid_post", test_fraction=0.2)
best = Probe.best_layer(probes, by="cohens_d_logistic")    # not auc_test_logistic
print(f"best layer = {best.layer} (depth {best.normalized_depth:.3f})")
print(f"hook_name  = {best.hook_name}")
print(f"directions stored: {list(best.directions)}")
print()
print("metrics on best layer:")
for k, v in sorted(best.metrics.items()):
    print(f"  {k:35s} {v:.4f}" if isinstance(v, float) else f"  {k:35s} {v}")

## Phase 5 — visualize the layer-selection curve

Cohen's d shows how cleanly each layer separates positive from negative classes (in standard-deviation units of the logistic decision function). On a 100-pair dataset with a 20% test split, the held-out AUC saturates at 1.0 across many layers because `d_model = 1536` ≫ `n_test = 20-vs-20` makes any class boundary trivially perfect. Cohen's d keeps growing as the gap widens — so it's the more informative y-axis when AUC has plateaued.

The x-axis here is normalized depth: 0.0 = embedding output, 1.0 = post-final-layernorm. Useful when comparing across models with different layer counts.


In [ ]:
from steerkit import plot_layer_selection

plot_layer_selection(probes, by_classifier="cohens_d_logistic", x_axis="normalized_depth", title="refusal probe across layers")

## Phase 6 — calibrate α

Steering strength α controls how much we push the activation toward the concept direction: `act ← act + α·v`. Too small, no effect. Too big, the model goes out of distribution and outputs gibberish. `calibrate_alpha` automates the choice:

1. For each candidate α ∈ {0.5, 1, 2, 4, 8} (configurable):
   1. Generate a steered completion with the steering hook installed at strength α.
   2. Compute that completion's perplexity *under the unsteered forward pass* — "how surprised is the original model by what the steered version produced?"
2. Compute `ratio = mean(steered ppl) / mean(unsteered ppl)`.
3. Pick the **largest α** whose ratio stays under a ceiling (default 1.5×). That's the strongest perturbation that still keeps the model coherent.

Result attaches to `probe.auto_alpha`; `Probe.steer(model, prompt)` uses it when `alpha=None`.

In [ ]:
from steerkit import calibrate_alpha

# Defaults: 4 calibration prompts × 30 max-new-tokens. ~30-60s on Qwen2.5-1.5B / MPS.
chosen, ratios = calibrate_alpha(best, model)
print(f"auto-α = {chosen}")
for a in sorted(ratios):
    note = "  ← chosen" if a == chosen else ""
    over = " (over ceiling)" if ratios[a] > 1.5 else ""
    print(f"  α={a:>4}: ppl ratio = {ratios[a]:.3f}{note}{over}")


## Phase 7 — save and reload

The probe artifact is a single `.probe.safetensors` file: tensors (the three direction vectors + biases) plus an embedded JSON metadata blob (`schema_version`, `model.id`, `hook_name`, `layer`, `normalized_depth`, `auto_alpha`, all the metrics, etc.).

Reloading needs nothing beyond the path — no original training infrastructure, no original dataset, just `Probe.load(...)`. That portability is the v0.1 cross-model differentiator.

In [ ]:
best.save(REPO_ROOT / "runs" / "refusal.probe.safetensors")
reloaded = Probe.load(REPO_ROOT / "runs" / "refusal.probe.safetensors")
print(f"reloaded probe trained on {reloaded.model_id}")
print(f"  layer: {reloaded.layer} (depth {reloaded.normalized_depth:.3f})")
print(f"  auto_alpha: {reloaded.auto_alpha}")
print(f"  schema: v{reloaded.schema_version}")

## Phase 8 — steer

Four operations are supported on `Probe.steer(..., op=...)`:

* **addition** (default): `act + α·v` — push toward the concept
* **projection**: `act − (act·v̂)v̂` — remove the concept entirely
* **clamp**: `act + (target − act·v̂)v̂` — force the projection to a target value
* **multiplicative** (amplify): `act + (γ−1)(act·v̂)v̂` — scale existing signal

Convenience methods `probe.ablate(...)`, `probe.clamp(target=...)`, `probe.amplify(gamma=...)` wrap the corresponding ops.

**The empirical reality** of behavioral steering: `calibrate_alpha`'s perplexity-ceiling target (1.5× unsteered) is conservative — protecting coherence, not behavioral commitment. On a 1.5B chat model trained on no-help refusal pairs, you'll typically see auto-α produce *disclaimer-prefixed help* ("As an AI language model, I cannot... however, here are some suggestions: ..."), and **canonical no-help refusals** ("I'm sorry, but I can't assist with that.") fire at roughly 1.5-2× auto-α. Below we test both.


In [ ]:
BENIGN_PROMPTS = [
    "What's a good way to start the morning?",
    "Recommend a book for a long flight.",
    "How do I bake bread?",
]
# Refusal detection — only canonical no-help refusals.
# Deliberately NOT matching "As an AI" / "As a language model" prefixes:
# those are chat-tuning's *helpful* preamble (the model says them and then
# answers the question anyway), not refusal. Empirically the model says
# "As an AI language model, I cannot recommend a specific book..." and then
# recommends books. Only the canonical no-help phrases below reliably indicate
# the model actually withheld the answer.
REFUSAL_PATTERNS = (
    # Canonical no-help refusals — what the dataset trained the probe to push toward.
    "i cannot", "i can't help", "i can't assist", "i can't provide",
    "i'm not able", "i won't", "i would not", "i don't think i can",
    "i'm unable", "i'd rather not", "i decline", "i refuse",
    "i'm afraid i can", "i apologize, but", "i'm sorry, but i",
    # NB: deliberately NOT matching "as an AI" / "as a language model" / etc. —
    # those are chat-tuning's standard *helpful* preface, not refusal. The model
    # says "As an AI language model, I cannot recommend a specific book..." and
    # then recommends books anyway. Only the canonical no-help phrases above are
    # reliable indicators that the model actually refused.
)
FALSE_POSITIVE_PATTERNS = ("sorry to hear",)  # sympathetic, not refusal


def looks_like_refusal(text: str) -> bool:
    low = text.lower()
    if any(fp in low for fp in FALSE_POSITIVE_PATTERNS):
        return False
    return any(p in low for p in REFUSAL_PATTERNS)


def show(label, text, prompt):
    flag = "  ⚠ REFUSAL-LIKE" if looks_like_refusal(text) else ""
    # Print label + flag on one line, full text indented below — easier to scan
    # multi-sentence outputs than a 120-char repr slice would be.
    print(f"  {label}:{flag}")
    for line in text.splitlines() or [text]:
        print(f"      {line}")


# 1) Auto-α at the classifier-best layer (the layer fit_all picked).
MAX_TOKENS = 512   # longer than 40 so deflection-style refusals have room to complete
print(f"=== classifier-best layer = {reloaded.layer} (depth {reloaded.normalized_depth:.2f}), auto-α = {reloaded.auto_alpha:.1f} ===\n")
for prompt in BENIGN_PROMPTS:
    print(f"prompt: {prompt}")
    show("unsteered", reloaded.steer(model, prompt, alpha=0.0, max_new_tokens=MAX_TOKENS), prompt)
    show(f"α=auto({reloaded.auto_alpha:.1f})", reloaded.steer(model, prompt, max_new_tokens=MAX_TOKENS), prompt)
    # Auto-α is conservative (perplexity ceiling 1.5×). Canonical no-help refusals
    # typically fire at 1.5-2× auto on a 1.5B chat model trained on no-help pairs.
    boost = 2.0 * reloaded.auto_alpha
    show(f"α=2×auto({boost:.1f})", reloaded.steer(model, prompt, alpha=boost, max_new_tokens=MAX_TOKENS), prompt)
    print()


### Phase 8b — sweep α at the classifier-best layer

If auto-α didn't flip behavior at the classifier-best layer, push harder. The perplexity ceiling protects coherence, not behavioral change — a higher α may produce refusal even at higher perplexity.


In [ ]:
test_prompt = BENIGN_PROMPTS[0]
print(f"prompt: {test_prompt}\n")
show("unsteered", reloaded.steer(model, test_prompt, alpha=0.0, max_new_tokens=MAX_TOKENS), test_prompt)
for alpha in [4, 8, 16, 32, 64]:
    out = reloaded.steer(model, test_prompt, alpha=alpha, max_new_tokens=MAX_TOKENS)
    show(f"α={alpha}", out, test_prompt)


### Phase 8c — try a middle-network layer (with α scaled to the residual norm)

On Qwen2.5-1.5B-Instruct the `cohens_d_logistic`-best layer happens to be near the middle (around depth 0.5), so this section is more illustration than fix. **The principle still matters when you generalize to other models**: pre-LayerNorm transformers (Qwen / Llama / Gemma / GPT-NeoX — almost everything modern) accumulate residual stream norm through depth without re-normalizing between blocks. Only `ln_final.hook_normalized` has a small controlled norm; intermediate layers' residual streams can be 100-1000× larger.

**The footgun:** the same numeric `α=8` that's a 30% perturbation at the post-final-layernorm output is invisible (~0.16% perturbation) at a deep mid-network layer. Always scale α to the layer's activation norm; that's what `metrics["activation_norm_mean"]` is for, and what `calibrate_alpha` uses internally.

Below we measure the norm and parameterize α as a fraction of it.


In [ ]:
from steerkit import calibrate_alpha

block_layers = sorted(layer for layer in probes if 0 <= layer < model.n_layers)
mid_layer = block_layers[len(block_layers) // 2]
mid_probe = probes[mid_layer]

mid_acts = activations[mid_layer]
mid_norm = float(mid_acts.norm(dim=-1).mean())
print(f"=== middle layer = {mid_probe.layer} (depth {mid_probe.normalized_depth:.2f}, cohens_d={mid_probe.metrics['cohens_d_logistic']:.2f}) ===")
print(f"typical residual norm at this layer: {mid_norm:.0f}\n")

# Sweep α as fractions of the layer norm.
FRACTIONS = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.25, 1.5]
for prompt in BENIGN_PROMPTS:
    print(f"prompt: {prompt}")
    show("unsteered", mid_probe.steer(model, prompt, alpha=0.0, max_new_tokens=MAX_TOKENS), prompt)
    for frac in FRACTIONS:
        alpha = frac * mid_norm
        show(f"α={alpha:.0f} ({frac*100:.0f}% of norm)", mid_probe.steer(model, prompt, alpha=alpha, max_new_tokens=MAX_TOKENS), prompt)
    print()


### What just happened

The α sweep above shows the **gradient of behavioral steering** as α grows. With the no-help-trained probe on Qwen2.5-1.5B-Instruct, the four regimes you see are:

1. **At α ≤ auto-α (≈ ppl ratio 1.5):** style drift toward AI-disclaimer prefaces ("As an AI language model, ...") but the model **still answers helpfully** afterwards. The refusal direction is biasing the prefix tokens but not the actual response content. Don't be misled by the prefix.
2. **At α ≈ 1.5-2× auto-α:** **canonical no-help refusals fire** — *"I'm sorry, but I can't assist with that."* The model actually withholds the answer. This is the regime the dataset was designed to push toward.
3. **At α ≳ 2.5× auto-α:** coherence starts to break down — repetitive loops, off-topic drift, or short fragments.
4. **At α ≳ 3-4× auto-α:** gibberish.

Why is auto-α "too low" for canonical refusals? `calibrate_alpha`'s default ceiling is `steered_ppl / unsteered_ppl ≤ 1.5`, which protects coherence but not behavioral commitment. Crossing into actual refusal often pushes perplexity to the 1.5-2.5 range — the steered text is lower-probability under the unsteered distribution but still grammatical. Multiply auto-α by 2 if you want canonical no-help outputs reliably.


### Phase 8d — the other three operations (briefly)

Once you've found a working (layer, α) for `addition`, the other three ops are mechanically the same:


In [ ]:
test_prompt = BENIGN_PROMPTS[0]
alpha_for_addition = 0.25 * mid_norm  # ≈ 1.5× auto-α at this layer norm — the regime where canonical refusals fire
show("unsteered", mid_probe.steer(model, test_prompt, alpha=0.0, max_new_tokens=MAX_TOKENS), test_prompt)
show("addition", mid_probe.steer(model, test_prompt, alpha=alpha_for_addition, max_new_tokens=MAX_TOKENS), test_prompt)
show("ablate", mid_probe.ablate(model, test_prompt, max_new_tokens=MAX_TOKENS), test_prompt)
show("clamp(2.0)", mid_probe.clamp(model, test_prompt, target=2.0, max_new_tokens=MAX_TOKENS), test_prompt)
show("amplify(2)", mid_probe.amplify(model, test_prompt, gamma=2.0, max_new_tokens=MAX_TOKENS), test_prompt)


### Phase 8e — where in the response does the direction fire?

Held-out AUC tells you the probe separates positive from negative *responses*, but it collapses the entire response to one last-token vector. To see *where in a generated sequence* the refusal direction is most active, project every token's residual stream onto the probe direction with `probe.score_tokens(...)`.

What you're looking at:

* **Bars to the right of zero (red)** — the refusal direction is *firing* at that token (concept active).
* **Bars to the left (blue)** — the direction is *quiet* (concept inactive).
* **Magnitude** — how strongly the direction fires *relative to other positions in the same response*. Across-response comparisons aren't calibrated; within-response comparisons are.

If the unsteered helpful response is mostly blue and the steered refusal is mostly red on tokens like *can't*, *won't*, or end-of-turn, the steering hook is doing what the probe says it should. If the bars are loudest on punctuation or filler, the probe is keying on something other than refusal — a sign to tighten the dataset.


In [ ]:
# 1. Generate one unsteered + one steered response on the same prompt.
prompt = "What's a fun activity for a quiet Sunday afternoon?"
MAX_VIZ_TOKENS = 18  # short enough to plot per-token without scrolling

unsteered_text = best.steer(model, prompt, alpha=0.0, max_new_tokens=MAX_VIZ_TOKENS)
# 2× the auto-calibrated α is past the perplexity ceiling but produces a clearer
# behavioral flip — handy for a visualization. Drop to ~best.auto_alpha for a more
# coherence-preserving steered output.
steered_alpha = (best.auto_alpha or 2.0) * 2.0
steered_text = best.steer(model, prompt, alpha=steered_alpha, max_new_tokens=MAX_VIZ_TOKENS)

print(f"unsteered (α=0): {unsteered_text}")
print(f"steered   (α={steered_alpha:.2g}): {steered_text}")
print()

# 2. Score every token of each response.
unsteered_scores = best.score_tokens(model, prompt, unsteered_text)
steered_scores = best.score_tokens(model, prompt, steered_text)

# 3. Render side-by-side bar charts.
import matplotlib.pyplot as plt

fig, (ax_u, ax_s) = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, ts, label in (
    (ax_u, unsteered_scores, f"unsteered  ·  α=0.0"),
    (ax_s, steered_scores, f"steered  ·  α={steered_alpha:.2g}"),
):
    vals = ts.scores.float().cpu().numpy()
    colors = ["tab:red" if v >= 0 else "tab:blue" for v in vals]
    ax.barh(range(len(ts.tokens)), vals, color=colors)
    ax.invert_yaxis()
    ax.set_yticks(range(len(ts.tokens)))
    ax.set_yticklabels([repr(t) for t in ts.tokens], fontsize=8)
    ax.axvline(0, color="black", lw=0.5)
    ax.set_title(label)
    ax.set_xlabel(f"projection onto refusal direction (layer {best.layer})")
fig.suptitle(f"Per-token probe scores  ·  prompt: {prompt!r}")
fig.tight_layout()
plt.show()


Two complementary views of the same direction:

* `plot_logit_lens(probe, model)` — **what *vocabulary* the direction promotes** (refusal-style tokens regardless of context).
* `probe.score_tokens(...)` — **where *in this specific response* the direction fires** (positions, in context).

The first answers "is this direction about refusal at all?"; the second answers "is this generation actually expressing the refusal direction, and where?".


## Phase 9 — evaluate

Held-out AUC + Cohen's d are already attached to `probe.metrics` from Phase 4. The `steerkit.eval` module adds a few more metrics you can run after fitting:

* **`logit_lens_vocab_score`** — push the direction through the unembed and check what fraction of the top-K vocabulary tokens are in a user-supplied target set. The cheapest interpretability sanity check.
* **`perplexity_ratio`** — the same metric `calibrate_alpha` uses for its ceiling, exposed standalone so you can ask "at this α, how much coherence did I cost?"
* **`external_classifier_score`** — generate steered + unsteered completions, run any user-supplied classifier function, return the mean shift.
* **`evaluate_steering_effect`** — the LLM-judge expensive tier (Phase 3): teacher rates each steered completion on a 1-7 scale.

`evaluate_probe(...)` is a one-shot aggregator that runs whichever inputs you supply and returns an `EvalReport`.

In [ ]:
from steerkit import evaluate_probe

REFUSAL_VOCAB = {"sorry", "apologize", "afraid", "unable", "cannot", "can't", "won't", "decline", "refuse"}
report = evaluate_probe(
    reloaded,
    model,
    target_vocab=REFUSAL_VOCAB,
    perplexity_prompts=["Tell me about your morning.", "Recommend a book."],
)
print("summary:", report.summary())
print()
print("cheap metrics:", {k: round(v, 3) for k, v in report.cheap.items() if isinstance(v, (int, float))})
print("logit-lens top tokens:", report.logit_lens["top_k_tokens"][:8])
print("  ↳ refusal-vocab matches:", report.logit_lens["matches"])
print("perplexity ratio at auto-α:", round(report.perplexity["ratio"], 3))

## What's next

* **`quickstart_refusal.ipynb`** — same content, compressed.
* **`quickstart_emotion.ipynb`** — multi-class `ConceptGroup` (joy / sadness / anger), multinomial diagnostic probe, similarity heatmap.
* **`quickstart_composition.ipynb`** — `compose([verbose_probe, formal_probe])` for cross-group simultaneous steering.

Or jump to the full design memory in [`docs/design.md`](../docs/design.md).